# FactIndic final pilot: Roman-script Hinglish news summarization and factuality

This is the final corrected experiment for the Roman-script Hinglish scope. It uses the public **News_Hinglish_English** corpus: English news content paired with Romanized Hinglish versions. The corpus is labelled for translation and its Hinglish side is machine-generated then curated; it does **not** contain gold summaries. We therefore do not report ROUGE or treat the English parallel article as a summary reference.

Study: 30 fixed Roman-script Hinglish news articles × 2 model sizes × 2 prompt styles = 120 summaries. The completed factuality signal is an AI-assisted source-grounded screen. Human factuality validation has been skipped for this version. The two models are Qwen3 1.7B and Qwen3 4B, so the model comparison is within one family and mainly reflects scale. Results are exploratory and limited by the dataset’s machine-generated Hinglish text and news domain.

The source dataset is pinned to an immutable Hugging Face revision and is not copied into this public repository. Source text, generated summaries, the blind key, and review sheets remain in your private Colab Drive. The downloaded pack contains source text, so keep it private and never commit it to GitHub.


## 1. Start a Colab GPU runtime and install the inference tools

Choose **Runtime → Change runtime type → GPU**. This notebook runs inference in Colab; it does not train the models or load them on your Mac.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece pyarrow

## 2. Set up the private workspace

In [ ]:
import csv, gc, hashlib, io, json, random, re, time, zipfile
from collections import Counter, defaultdict
from statistics import mean, median
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import torch
import transformers
from google.colab import drive, files
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > GPU, then rerun from the top."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount('/content/drive')
PRIVATE_ROOT = Path('/content/drive/MyDrive/FactIndic_Private')
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'suyash2739/News_Hinglish_English'
DATASET_REVISION = '22a19b81fd8cc151a9d011cfdd7e6dd4e9f2b359'
PARQUET_URL = (
    'https://huggingface.co/datasets/' + DATASET_ID +
    '/resolve/' + DATASET_REVISION + '/data/train-00000-of-00001.parquet'
)


## 3. Download, audit, and select the fixed sample

The code checks that the source is Latin-script Hinglish, not Devanagari Hindi with English tokens. It also checks for blank or duplicate pairs, records the dataset hash, and selects a reproducible 30-row sample.

In [ ]:
response = requests.get(PARQUET_URL, timeout=120)
response.raise_for_status()
parquet_bytes = response.content
DATA_SHA256 = hashlib.sha256(parquet_bytes).hexdigest()
df = pd.read_parquet(io.BytesIO(parquet_bytes))
required = {'content', 'Hinglish'}
if not required.issubset(df.columns):
    raise ValueError(f"Unexpected dataset columns: {list(df.columns)}")

DEV = re.compile(r'[\u0900-\u097F]')
LATIN = re.compile(r'[A-Za-z]')

def clean_text(value):
    return re.sub(r'\s+', ' ', str(value)).strip()

df['content'] = df['content'].map(clean_text)
df['Hinglish'] = df['Hinglish'].map(clean_text)
blank = (df['content'].eq('') | df['Hinglish'].eq(''))
dev_rows = df['Hinglish'].map(lambda x: bool(DEV.search(x)))
latin_rows = df['Hinglish'].map(lambda x: bool(LATIN.search(x)))
replacement_rows = df['content'].str.contains('�', regex=False) | df['Hinglish'].str.contains('�', regex=False)
duplicate_pairs = df.duplicated(subset=['content','Hinglish'], keep='first')
print('Dataset rows:', len(df))
print('Rows with Latin letters in Hinglish:', int(latin_rows.sum()))
print('Rows with Devanagari in Hinglish:', int(dev_rows.sum()))
print('Blank pairs:', int(blank.sum()))
print('Rows with corrupted replacement characters:', int(replacement_rows.sum()))
print('Duplicate pairs after first occurrence:', int(duplicate_pairs.sum()))
print('Dataset SHA-256:', DATA_SHA256)

eligible = df.index[(~blank) & (~dev_rows) & latin_rows & (~replacement_rows) & (~duplicate_pairs)].tolist()
if len(eligible) < 30:
    raise ValueError(f'Only {len(eligible)} Roman-script source pairs passed the automatic audit.')
SEED = 42
N_SAMPLE = 30
selected_indices = sorted(random.Random(SEED).sample(eligible, N_SAMPLE))
sample = []
for i in selected_indices:
    sample.append({
        'source_id': f'news_{int(i):05d}',
        'source': df.at[i, 'Hinglish'],
        'parallel_english': df.at[i, 'content'],
    })
print(f'Selected {len(sample)} rows, seed={SEED}; source truncation will be checked per model.')


## 4. Inspect the selected Hinglish examples

Check that the sample contains Hindi words written in Latin letters mixed with English, in the style intended for the project. The parallel English article is the same news content, **not** a reference summary. If the sample includes Devanagari or is not actually code-mixed, stop and send the audit counts and examples for correction.

In [ ]:
from IPython.display import display
preview_rng = random.Random(SEED + 1)
preview_positions = sorted(preview_rng.sample(range(len(sample)), min(8, len(sample))))
display(pd.DataFrame([{
    'source_id': sample[i]['source_id'],
    'Hinglish input': sample[i]['source'],
    'parallel English article (not a summary)': sample[i]['parallel_english'],
} for i in preview_positions]))


## 5. Confirm the sample and configure the final run

After inspecting the preview, set `SAMPLE_LANGUAGE_CHECKED = True`. Teacher approval for using any dataset is already recorded. The run compares two Qwen3 sizes under matched decoding settings; this avoids the separate gated Gemma model download.

In [ ]:
SAMPLE_LANGUAGE_CHECKED = False  # Set True only after you inspect Cell 4's preview.
TEACHER_APPROVED_DATASET = True
assert SAMPLE_LANGUAGE_CHECKED, 'Inspect the selected Hinglish examples before generation.'
assert TEACHER_APPROVED_DATASET

MODEL_IDS = {
    'qwen3_1_7b': 'Qwen/Qwen3-1.7B',
    'qwen3_4b': 'Qwen/Qwen3-4B',
}
PROMPTS = {
    'standard': (
        'Summarize this Roman-script Hindi-English news article in clear, concise English. '
        'Preserve the central event, people, places, numbers, and stated outcomes. '
        'Use only information in the article.'
    ),
    'factuality_aware': (
        'Write a concise English summary of this Roman-script Hindi-English news article. '
        'Use only facts explicitly stated in the article. Do not infer causes, intentions, '
        'dates, quantities, identities, or outcomes. Preserve who did what; omit unclear details.'
    ),
}
MAX_SOURCE_TOKENS = 1024
MAX_NEW_TOKENS_TRIES = (192, 256, 320)
DTYPE = torch.float16

fingerprint_payload = {
    'dataset_id': DATASET_ID, 'dataset_revision': DATASET_REVISION,
    'dataset_sha256': DATA_SHA256, 'sample_ids': [x['source_id'] for x in sample],
    'seed': SEED, 'models': MODEL_IDS, 'prompts': PROMPTS,
}
RUN_ID = 'final_hinglish_news_' + hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode()
).hexdigest()[:12]
RUN_DIR = PRIVATE_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
GENERATIONS_PATH = RUN_DIR / 'generations.jsonl'
print('Private run folder:', RUN_DIR)
print('Final comparison:', len(sample), 'sources ×', len(MODEL_IDS), 'models ×', len(PROMPTS), 'prompt arms')


## 6. Run the paired final generation

The models load one at a time to fit a Colab T4. Completed outputs are checkpointed to private Drive and skipped if the notebook reconnects. If a summary reaches the token cap, it is retried with a higher cap before being recorded.

In [ ]:
def load_existing(path):
    found = {}
    if path.exists():
        with path.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    # Bad checkpoint rows are deliberately retried on rerun.
                    if (row.get('hit_new_token_cap') or not row.get('prediction','').strip()
                            or '�' in row.get('prediction','')):
                        continue
                    key = (row['source_id'], row['model_key'], row['prompt_arm'])
                    found[key] = row
    return found

results = load_existing(GENERATIONS_PATH)
total = len(sample) * len(MODEL_IDS) * len(PROMPTS)
print(f'{len(results)}/{total} outputs already checkpointed')

for model_key, model_id in MODEL_IDS.items():
    needed = [(item, arm, instruction) for item in sample for arm, instruction in PROMPTS.items()
              if (item['source_id'], model_key, arm) not in results]
    if not needed:
        print(model_key, 'already complete; skipping model load')
        continue
    print('Loading', model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map='auto'
    ).eval()
    for item, arm, instruction in needed:
        source_ids = tokenizer.encode(item['source'], add_special_tokens=False)
        truncated = len(source_ids) > MAX_SOURCE_TOKENS
        source_text = tokenizer.decode(source_ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
        prompt = (
            f'{instruction}\n\nArticle (Roman-script Hinglish):\n{source_text}'
            '\n\nEnglish summary:'
        )
        messages = [{'role':'user', 'content':prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt', enable_thinking=False
        ).to(model.device)
        input_length = int(inputs['input_ids'].shape[-1])
        started = time.time()
        prediction = ''
        generated_tokens = 0
        attempts = 0
        hit_cap = False
        for cap in MAX_NEW_TOKENS_TRIES:
            attempts += 1
            with torch.inference_mode():
                output = model.generate(
                    **inputs, max_new_tokens=cap, do_sample=False, num_beams=1,
                    repetition_penalty=1.03, no_repeat_ngram_size=3,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            new_ids = output[0][input_length:]
            generated_tokens = int(len(new_ids))
            hit_cap = generated_tokens >= cap
            prediction = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            if not hit_cap:
                break
        row = {
            'source_id':item['source_id'], 'source':item['source'],
            'parallel_english':item['parallel_english'], 'model_key':model_key,
            'model_id':model_id, 'prompt_arm':arm, 'prediction':prediction,
            'source_tokens_before_truncation':len(source_ids),
            'source_truncated':bool(truncated), 'input_tokens_with_prompt':input_length,
            'generated_tokens':generated_tokens, 'generation_attempts':attempts,
            'hit_new_token_cap':bool(hit_cap), 'inference_seconds':round(time.time()-started,3),
        }
        results[(item['source_id'],model_key,arm)] = row
        with GENERATIONS_PATH.open('a',encoding='utf-8') as f:
            f.write(json.dumps(row,ensure_ascii=False)+'\n')
            f.flush()
        if len(results) % 8 == 0 or len(results) == total:
            print(f'{len(results)}/{total} outputs saved')
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

if len(results) != total:
    raise RuntimeError(f'Incomplete run: {len(results)}/{total} outputs. Rerun this cell to resume.')
rows = list(results.values())
print('Generation complete:', len(rows), 'outputs')


## 7. Quality gate, generation metrics, and private review sheet

Mechanical checks do not determine factuality. A clean run exports per-condition summary length, source length, compression ratio, generation time, truncation counts, and a private blinded worksheet. The worksheet is optional for this project version. If empty outputs, replacement characters, token-cap hits, or duplicate condition keys remain, the export is marked diagnostic-only.


In [ ]:
expected = {(s['source_id'],m,a) for s in sample for m in MODEL_IDS for a in PROMPTS}
actual = {(r['source_id'],r['model_key'],r['prompt_arm']) for r in rows}
if actual != expected:
    raise RuntimeError(f'Condition mismatch: missing={len(expected-actual)}, extra={len(actual-expected)}')
terminal_marks = ('.','!','?','…','”','"')
cap_hits = sum(bool(r['hit_new_token_cap']) for r in rows)
empty_rows = sum(not r['prediction'].strip() for r in rows)
replacement_rows = sum('�' in r['prediction'] for r in rows)
no_terminal = sum(not r['prediction'].rstrip().endswith(terminal_marks) for r in rows)
status = 'diagnostic_only' if (cap_hits or empty_rows or replacement_rows) else 'annotation_candidate'
audit = {
    'status':status,'rows':len(rows),'cap_hits':cap_hits,'empty_rows':empty_rows,
    'replacement_char_rows':replacement_rows,'without_terminal_punctuation':no_terminal,
    'source_truncated_rows':sum(bool(r['source_truncated']) for r in rows),
    'reference_metrics':'not_applicable_no_gold_summaries',
    'automatic_checks_are_not_factuality_labels':True,
}
(RUN_DIR/'quality_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
print(audit)

# Paper-ready descriptive generation metrics for each model/prompt condition.
# Word counts use whitespace tokenization; compression is summary words / source words.
def whitespace_words(text):
    return len(str(text).split())

metrics = []
for model_key in MODEL_IDS:
    for arm in PROMPTS:
        group = [r for r in rows if r['model_key'] == model_key and r['prompt_arm'] == arm]
        summary_word_counts = [whitespace_words(r['prediction']) for r in group]
        source_word_counts = [whitespace_words(r['source']) for r in group]
        per_doc_ratios = [s / src for s, src in zip(summary_word_counts, source_word_counts) if src]
        total_source_words = sum(source_word_counts)
        metrics.append({
            'model_key': model_key, 'prompt_arm': arm, 'n_outputs': len(group),
            'mean_summary_words': round(mean(summary_word_counts), 2),
            'median_summary_words': round(median(summary_word_counts), 2),
            'mean_source_words': round(mean(source_word_counts), 2),
            'mean_per_document_compression_ratio': round(mean(per_doc_ratios), 4) if per_doc_ratios else None,
            'corpus_compression_ratio': round(sum(summary_word_counts) / total_source_words, 4) if total_source_words else None,
            'mean_generation_seconds': round(mean(r['inference_seconds'] for r in group), 3),
            'median_generation_seconds': round(median(r['inference_seconds'] for r in group), 3),
            'source_truncated_n': sum(bool(r['source_truncated']) for r in group),
            'mean_generated_tokens': round(mean(r['generated_tokens'] for r in group), 2),
        })
metrics_path = RUN_DIR / 'generation_metrics.csv'
with metrics_path.open('w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(metrics[0]))
    w.writeheader(); w.writerows(metrics)
print('Paper-ready descriptive metrics (not factuality labels):')
for row in metrics: print(row)

if status == 'diagnostic_only':
    raise RuntimeError('Diagnostic-only run. Do not treat these outputs as the final annotation candidate.')

rng = random.Random(SEED + 99)
blind_rows=[]
private_key=[]
for r in rows:
    blind_id = hashlib.sha256(f"{RUN_ID}|{r['source_id']}|{r['model_key']}|{r['prompt_arm']}".encode()).hexdigest()[:12]
    blind_rows.append({
        'blind_id':blind_id,'source_hinglish':r['source'],
        'parallel_english_article_for_reading_aid_only':r['parallel_english'],
        'generated_summary':r['prediction'],'human_factuality':'','error_type':'',
        'unsupported_or_wrong_detail':'','annotator_id':'','notes':'',
    })
    private_key.append({'blind_id':blind_id,'source_id':r['source_id'],
                        'model_key':r['model_key'],'prompt_arm':r['prompt_arm']})
rng.shuffle(blind_rows)
review_path = RUN_DIR/'human_review_blinded.csv'
with review_path.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(blind_rows[0]));w.writeheader();w.writerows(blind_rows)
with (RUN_DIR/'private_unblinding_key.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(private_key[0]));w.writeheader();w.writerows(private_key)
metadata = {
    'run_id':RUN_ID,'status':status,'dataset_id':DATASET_ID,
    'dataset_revision':DATASET_REVISION,'dataset_sha256':DATA_SHA256,
    'dataset_has_reference_summaries':False,'n_sources':len(sample),'n_generations':len(rows),
    'models':MODEL_IDS,'prompt_arms':list(PROMPTS),'seed':SEED,
    'max_source_tokens':MAX_SOURCE_TOKENS,'new_token_caps_tried':MAX_NEW_TOKENS_TRIES,
    'decoding':'greedy, do_sample=False, num_beams=1','gpu':torch.cuda.get_device_name(0),
    'transformers_version':transformers.__version__,'torch_version':torch.__version__,
    'human_review_status':'not_started','factuality_needs_source_grounded_human_labels':True,
}
(RUN_DIR/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
private_zip = PRIVATE_ROOT/'factindic_final_hinglish_private_review.zip'
with zipfile.ZipFile(private_zip,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(review_path,arcname='human_review_blinded.csv')
    z.write(RUN_DIR/'quality_audit.json',arcname='quality_audit.json')
    z.write(RUN_DIR/'run_metadata.json',arcname='run_metadata.json')
    z.write(metrics_path,arcname='generation_metrics.csv')
print('Private review pack:',private_zip)
files.download(str(private_zip))


## 8. Optional human factuality review

Human factuality review is skipped for the current project version. Stop after Section 7 if you only need the final generation metrics and quality audit. Do not present the AI-assisted screen as human-verified factuality.

If human review is completed later, check every Hinglish source and summary, use the four labels, record evidence/error spans, and enter the actual annotator ID. The parallel English article is only a reading aid.


In [ ]:
# Optional: import a completed human or AI-assisted review file.
# Default this to AI because the current project version skips human verification.
# Change to 'human_single_annotator' only after you personally judged every row.
REVIEW_TYPE = 'ai_assisted'
assert REVIEW_TYPE in {'ai_assisted', 'human_single_annotator'}

uploaded = files.upload()
review_name = next((name for name in uploaded if name.lower().endswith('.csv')), None)
if not review_name:
    raise ValueError('Upload a completed review CSV, or skip this optional section.')
filled_path = RUN_DIR / ('ai_review_filled.csv' if REVIEW_TYPE == 'ai_assisted' else 'human_review_filled.csv')
filled_path.write_bytes(uploaded[review_name])
with filled_path.open(encoding='utf-8-sig', newline='') as f:
    reviewed = list(csv.DictReader(f))
allowed = {'supported', 'minor_error', 'major_error', 'cannot_determine'}
required_cols = {'blind_id', 'human_factuality'}
if not reviewed or not required_cols.issubset(reviewed[0]):
    raise ValueError('Review CSV is missing required columns.')
labels = [r['human_factuality'].strip().lower() for r in reviewed]
if len(reviewed) != len(rows) or any(label not in allowed for label in labels):
    raise ValueError('Every expected row needs one permitted factuality label.')
if len({r['blind_id'] for r in reviewed}) != len(reviewed):
    raise ValueError('Duplicate blind IDs found.')
if REVIEW_TYPE == 'ai_assisted':
    review_type = 'ai_assisted'
    for row in reviewed:
        row['annotator_id'] = 'AI_assistant'
else:
    reviewer_ids = {r.get('annotator_id', '').strip() for r in reviewed}
    if len(reviewer_ids) != 1 or '' in reviewer_ids or 'AI_assistant' in reviewer_ids:
        raise ValueError('Human review requires one real, nonblank annotator ID.')
    review_type = 'human_single_annotator'
    if len({r['human_factuality'].strip().lower() for r in reviewed}) == 1 and reviewed[0]['human_factuality'].strip().lower() == 'supported':
        print('Check that every row was individually reviewed; do not fill labels by default.')
with (RUN_DIR / 'private_unblinding_key.csv').open(encoding='utf-8-sig', newline='') as f:
    key = {r['blind_id']: r for r in csv.DictReader(f)}
if set(r['blind_id'] for r in reviewed) != set(key):
    raise ValueError('Review IDs do not match the private key.')
reviewed = [{**r, 'human_factuality': r['human_factuality'].strip().lower()} for r in reviewed]
print(f'Imported {review_type} review file. AI judgments remain AI judgments; they are not human factuality labels.')


## 9. Optional review analysis and aggregate export

Run this section only after importing a complete review file in Section 8. It labels AI-assisted review separately from human annotation and calculates paired bootstrap comparisons by source. The generation metrics were already exported in Section 7. This section does not create ROUGE scores because the dataset has no gold summaries. Skip it if you are not adding a review file.


In [ ]:
from collections import Counter
with (RUN_DIR/'private_unblinding_key.csv').open(encoding='utf-8-sig',newline='') as f:
    key={r['blind_id']:r for r in csv.DictReader(f)}
joined=[]
for r in reviewed:
    meta=key[r['blind_id']]
    joined.append({**r,**meta,'is_error':r['human_factuality'] in {'minor_error','major_error'}})

summary=[]
for model_key in MODEL_IDS:
    for arm in PROMPTS:
        group=[r for r in joined if r['model_key']==model_key and r['prompt_arm']==arm]
        counts=Counter(r['human_factuality'] for r in group)
        determinate=[r for r in group if r['human_factuality']!='cannot_determine']
        errs=sum(r['is_error'] for r in determinate)
        summary.append({
            'model_key':model_key,'prompt_arm':arm,'n':len(group),
            **{k:counts[k] for k in sorted(allowed)},
            'determinate_n':len(determinate),
            'minor_or_major_error_rate':round(errs/len(determinate),4) if determinate else None,
            'cannot_determine_rate':round(counts['cannot_determine']/len(group),4),
            'annotator_count':1 if review_type == 'human_single_annotator' else 0,
            'ai_reviewer_count':1 if review_type == 'ai_assisted' else 0,
            'review_type':review_type,
        })

def paired_bootstrap_error_diff(a,b,seed=42,B=5000):
    lookup={(r['source_id'],r['model_key'],r['prompt_arm']):r for r in joined}
    diffs=[]; excluded=0
    for s in sample:
        ra=lookup[(s['source_id'],*a)]
        rb=lookup[(s['source_id'],*b)]
        if 'cannot_determine' in (ra['human_factuality'],rb['human_factuality']):
            excluded+=1
        else:
            diffs.append(int(ra['is_error'])-int(rb['is_error']))
    if not diffs: return {'n_pairs':0,'excluded':excluded,'mean':None,'low':None,'high':None}
    rng=random.Random(seed)
    draws=sorted(sum(rng.choice(diffs) for _ in diffs)/len(diffs) for _ in range(B))
    return {'n_pairs':len(diffs),'excluded':excluded,'mean':sum(diffs)/len(diffs),
            'low':draws[int(.025*B)],'high':draws[int(.975*B)]}

paired=[]
for model_key in MODEL_IDS:
    x=paired_bootstrap_error_diff((model_key,'factuality_aware'),(model_key,'standard'))
    paired.append({'comparison':f'{model_key}: factuality-aware minus standard','n_pairs':x['n_pairs'],
                   'pairs_excluded':x['excluded'],'error_rate_difference':x['mean'],
                   'ci95_low':x['low'],'ci95_high':x['high']})
for arm in PROMPTS:
    x=paired_bootstrap_error_diff(('qwen3_1_7b',arm),('qwen3_4b',arm))
    paired.append({'comparison':f'Qwen3 1.7B minus Qwen3 4B: {arm}','n_pairs':x['n_pairs'],
                   'pairs_excluded':x['excluded'],'error_rate_difference':x['mean'],
                   'ci95_low':x['low'],'ci95_high':x['high']})

summary_name=('ai_assessed_factuality_summary.csv' if review_type=='ai_assisted' else 'human_factuality_summary.csv')
paired_name=('paired_ai_assessment_bootstrap.csv' if review_type=='ai_assisted' else 'paired_factuality_bootstrap.csv')
with (RUN_DIR/summary_name).open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(summary[0]));w.writeheader();w.writerows(summary)
with (RUN_DIR/paired_name).open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(paired[0]));w.writeheader();w.writerows(paired)
print(('AI-assessed screening counts (not human labels):' if review_type=='ai_assisted' else 'Human factuality counts:'))
for r in summary: print(r)
print('Paired comparisons:')
for r in paired: print(r)

aggregate_meta={
    'run_id':RUN_ID,'dataset_id':DATASET_ID,'dataset_revision':DATASET_REVISION,
    'n_sources':len(sample),'n_outputs':len(rows),'models':MODEL_IDS,
    'prompt_arms':list(PROMPTS),'seed':SEED,
    'dataset_hinglish_side_machine_generated_and_curated':True,
    'no_gold_summaries_no_rouge_reported':True,
    'review_type':review_type,
    'factuality_method':('AI-assisted source-grounded screening; not human annotation' if review_type=='ai_assisted' else 'one blinded source-grounded human annotator'),
    'human_review_status':('not_started' if review_type=='ai_assisted' else 'completed_single_annotator'),
    'human_factuality_claim_permitted':review_type=='human_single_annotator',
    'interpretation':('exploratory AI-assessed pilot; no human-verified factuality claim; news-domain distribution' if review_type=='ai_assisted' else 'exploratory pilot; single annotator; news-domain distribution'),
}
(RUN_DIR/'aggregate_metadata.json').write_text(json.dumps(aggregate_meta,ensure_ascii=False,indent=2),encoding='utf-8')
aggregate_zip=PRIVATE_ROOT/('factindic_final_hinglish_AI_assessed_aggregate_results.zip' if review_type=='ai_assisted' else 'factindic_final_hinglish_aggregate_results.zip')
with zipfile.ZipFile(aggregate_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['quality_audit.json','run_metadata.json','aggregate_metadata.json','generation_metrics.csv',summary_name,paired_name]:
        z.write(RUN_DIR/name,arcname=name)
    if review_type=='ai_assisted':
        z.writestr('AI_REVIEW_LIMITATIONS.txt','AI-assisted source-grounded screening only; not human annotation, gold factuality labels, or independently validated results. Human review remains not started. Do not present these counts as human factuality results.\n')
print('Aggregate archive:',aggregate_zip)
files.download(str(aggregate_zip))


## References and reporting limits

- Dataset card: [News_Hinglish_English](https://huggingface.co/datasets/suyash2739/News_Hinglish_English), DOI 10.57967/hf/5120. It provides English news articles and curated machine-generated Romanized Hinglish translations, not summary references.
- Qwen3 model cards: [1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) and [4B](https://huggingface.co/Qwen/Qwen3-4B). This is a within-family size comparison, not a comparison of different model families.
- Report the sample size, fixed sample seed, news-domain skew, and machine-generated Hinglish side. Use `generation_metrics.csv` for average summary/source length, compression, and inference time by model and prompt. Do not claim that model-based or automatic checks are human factuality labels. Do not report ROUGE without gold summaries. Human validation remains incomplete.
